In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report,r2_score,mean_squared_error
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures


In [24]:
df=pd.read_csv("../Datasets/created_house_price_prediction.csv")
df

,Area,Bedrooms,Bathrooms,Garage,YearBuilt,Location,Price
0,8000.0,3.0,2.0,1.0,2004.0,Lagos,105402678.0
1,8100.0,2.0,2.0,1.0,2013.0,Kaduna,62530695.0
2,8200.0,2.0,1.0,0.0,2023.0,Kano,71488309.0
3,8300.0,3.0,1.0,0.0,2006.0,Kano,70608894.0
4,8400.0,3.0,2.0,1.0,2003.0,Lagos,107029977.0
...,...,...,...,...,...,...,...
2045,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2046,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2047,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2048,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
df.isnull().sum()

Area         51
Bedrooms     51
Bathrooms    51
Garage       51
YearBuilt    51
Location     51
Price        51
dtype: int64

In [26]:
df.dropna(inplace=True)

In [27]:
df=pd.get_dummies(df)

In [28]:
x=df.drop("Price",axis=1)
y=df["Price"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [30]:
degrees = [1, 2, 3, 4, 5]
score = {"deg": [], "r2": [], "mse": []}

for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    
    x_train_poly = poly.fit_transform(x_train)
    x_test_poly = poly.transform(x_test) 
    model = LinearRegression()
    model.fit(x_train_poly, y_train)
    y_pred = model.predict(x_test_poly)
    score["deg"].append(degree)
    score["r2"].append(r2_score(y_test, y_pred))
    score["mse"].append(mean_squared_error(y_test, y_pred))
score

{'deg': [1, 2, 3, 4, 5],
 'r2': [0.21425046975387418,
  0.4962421250410982,
  0.7536180812103658,
  0.9075086934070644,
  0.922891790859132],
 'mse': [401114139847949.1,
  257161345858490.72,
  125774521810321.72,
  47215517743697.98,
  39362661756942.94]}

In [32]:
score_df=pd.DataFrame(score)
score_df

,deg,r2,mse
0,1,0.214250,4.011141e+14
1,2,0.496242,2.571613e+14
2,3,0.753618,1.257745e+14
3,4,0.907509,4.721552e+13
4,5,0.922892,3.936266e+13


In [43]:
best_deg=score_df[score_df["r2"]==score_df.r2.max()].deg.values[0]
best_deg

np.int64(5)

In [44]:
test_sizes=[0.05, 0.10, 0.15, 0.20, 0.25]
compare_dict={"model":[],"test_size":[],"r2":[],"mse":[]}
for test_size in test_sizes:
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=test_size,random_state=42)
    poly=PolynomialFeatures(best_deg)
    x_train_poly=poly.fit_transform(x_train)
    x_test_poly=poly.fit_transform(x_test)
    model=LinearRegression()
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    r2=r2_score(y_test,y_pred)
    mse=mean_squared_error(y_test,y_pred)
    compare_dict["model"].append("LinearRegression")
    compare_dict["test_size"].append(test_size)
    compare_dict["r2"].append(r2)
    compare_dict["mse"].append(mse)
    model.fit(x_train_poly,y_train)
    y_pred=model.predict(x_test_poly)
    r2=r2_score(y_test,y_pred)
    mse=mean_squared_error(y_test,y_pred)

    compare_dict["model"].append("Polynomial")
    compare_dict["test_size"].append(test_size)
    compare_dict["r2"].append(r2)
    compare_dict["mse"].append(mse)
compare_dict

{'model': ['LinearRegression',
  'Polynomial',
  'LinearRegression',
  'Polynomial',
  'LinearRegression',
  'Polynomial',
  'LinearRegression',
  'Polynomial',
  'LinearRegression',
  'Polynomial'],
 'test_size': [0.05, 0.05, 0.1, 0.1, 0.15, 0.15, 0.2, 0.2, 0.25, 0.25],
 'r2': [0.26864625805475495,
  0.9356090539332176,
  0.1789493462660503,
  0.9188048061596599,
  0.23028824412493298,
  0.9242589493521919,
  0.2142504697538461,
  0.922891790859132,
  0.17866445027248645,
  0.9207350221450246],
 'mse': [565504405722740.8,
  49788989378203.05,
  404787640491471.44,
  40030186669249.02,
  431647180772616.2,
  42474875472980.09,
  401114139847963.44,
  39362661756942.94,
  399457165860851.9,
  38550581935088.78]}

In [46]:
compare_df=pd.DataFrame(compare_dict)
compare_df

,model,test_size,r2,mse
0,LinearRegression,0.05,0.268646,5.655044e+14
1,Polynomial,0.05,0.935609,4.978899e+13
2,LinearRegression,0.10,0.178949,4.047876e+14
3,Polynomial,0.10,0.918805,4.003019e+13
4,LinearRegression,0.15,0.230288,4.316472e+14
5,Polynomial,0.15,0.924259,4.247488e+13
6,LinearRegression,0.20,0.214250,4.011141e+14
7,Polynomial,0.20,0.922892,3.936266e+13
8,LinearRegression,0.25,0.178664,3.994572e+14
9,Polynomial,0.25,0.920735,3.855058e+13


In [47]:
px.line(compare_df,x="test_size",y="r2",color="model")

In [48]:
px.line(compare_df,x="test_size",y="mse",color="model")